# Объединение спутниковых и метеоданных

Этот ноутбук создаёт единую таблицу для Шершнёвского водохранилища. Каждая строка — спутниковое наблюдение, а метеоданные подставляются на **тот же календарный день**.

Важно: это ещё не модель и не метка цветения. Здесь мы только готовим признаки: FAI и другие спутниковые показатели + температура, ветер, осадки и накопленные условия за 7/14/30 дней.

In [ ]:
from pathlib import Path
import pandas as pd

# Ноутбук нужно запускать из папки проекта hakaton.
SATELLITE_FILE = Path('shershnevskoe_multisensor_water_quality.csv')
WEATHER_FILE = Path('shershnevskoe_daily_weather_era5land.csv')
OUTPUT_FILE = Path('shershnevskoe_satellite_weather_merged.csv')

for file_path in [SATELLITE_FILE, WEATHER_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(f'Не найден файл: {file_path.resolve()}')

print('Спутники:', SATELLITE_FILE.resolve())
print('Погода:    ', WEATHER_FILE.resolve())

## 1. Читаем и проверяем исходные таблицы

`date` переводим в единый формат даты. В метеотаблице строго одна строка на день; в спутниковой таблице на один день могут приходиться наблюдения разных сенсоров — это нормально.

In [ ]:
satellite = pd.read_csv(SATELLITE_FILE)
weather = pd.read_csv(WEATHER_FILE)

satellite['date'] = pd.to_datetime(satellite['date']).dt.normalize()
weather['date'] = pd.to_datetime(weather['date']).dt.normalize()

if not weather['date'].is_unique:
    raise ValueError('В метеотаблице найдено больше одной строки на одну дату.')

print(f'Спутниковых наблюдений: {len(satellite):,}')
print(f'Период спутников: {satellite.date.min():%Y-%m-%d} — {satellite.date.max():%Y-%m-%d}')
print(f'Дней с погодой: {len(weather):,}')
print(f'Период погоды: {weather.date.min():%Y-%m-%d} — {weather.date.max():%Y-%m-%d}')

display(satellite[['date', 'sensor', 'FAI_mean', 'clear_water_frac', 'usable_for_fai']].head())

## 2. Объединяем по дате

Используется `left join`: ни одно спутниковое наблюдение не пропадёт. Если в один день есть Landsat и Sentinel-2, погода одного дня будет корректно добавлена к обеим строкам.

Из погоды исключаются только дублирующие служебные столбцы `year` и `month`: в спутниковой таблице они уже есть.

In [ ]:
weather_features = weather.drop(columns=['year', 'month'], errors='ignore')
merged = satellite.merge(
    weather_features,
    on='date',
    how='left',
    validate='many_to_one',
)

assert len(merged) == len(satellite), 'После объединения изменилось число спутниковых строк.'

weather_feature_columns = [col for col in weather_features.columns if col != 'date']
missing_weather_rows = merged[weather_feature_columns].isna().all(axis=1).sum()
print(f'Строк после объединения: {len(merged):,}')
print(f'Спутниковых строк без метеоданных: {missing_weather_rows:,}')

if missing_weather_rows:
    display(merged.loc[merged[weather_feature_columns].isna().all(axis=1), ['date', 'sensor']])
else:
    print('Готово: метеоданные найдены для каждого спутникового наблюдения.')

## 3. Смотрим результат

Полезные для первой модели столбцы:

- целевой спутниковый сигнал: `FAI_mean`, `FAI_p90`, `FAI_p95`;
- контроль качества снимка: `usable_for_fai`, `clear_water_frac`;
- температура дня: `air_temp_c`, `lake_mix_temp_c`, `lake_total_temp_c`;
- накопленные факторы до даты снимка: столбцы с суффиксами `prev_7d`, `prev_14d`, `prev_30d`.

Суффикс `prev_7d` означает предыдущие семь дней и не использует будущую информацию.

In [ ]:
preview_columns = [
    'date', 'sensor', 'FAI_mean', 'FAI_p90', 'clear_water_frac', 'usable_for_fai',
    'air_temp_c', 'lake_mix_temp_c', 'lake_total_temp_c',
    'wind_speed_10m_ms', 'precipitation_mm',
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d',
    'precipitation_sum_prev_7d', 'calm_days_prev_7d',
]
preview_columns = [column for column in preview_columns if column in merged.columns]
display(merged[preview_columns].sort_values('date').head(10))

print('Наблюдения по сенсорам:')
display(merged.groupby('sensor', dropna=False).agg(
    observations=('date', 'size'),
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    usable_fai_share=('usable_for_fai', 'mean'),
).reset_index())

## 4. Сохраняем общую базу

CSV сохраняется в текущую папку. Даты записываются в формате `ГГГГ-ММ-ДД`, чтобы файл было удобно открывать и в Python, и в Excel.

In [ ]:
merged_to_save = merged.sort_values(['date', 'sensor']).copy()
merged_to_save['date'] = merged_to_save['date'].dt.strftime('%Y-%m-%d')
merged_to_save.to_csv(OUTPUT_FILE, index=False)

print(f'Сохранено: {OUTPUT_FILE.resolve()}')
print(f'Размер общей таблицы: {merged_to_save.shape[0]:,} строк × {merged_to_save.shape[1]} столбцов')

# EDA: первичный анализ данных

Здесь мы проверяем, достаточно ли качественных спутниковых наблюдений, как ведёт себя FAI во времени и с какими метеофакторами он связан. Это исследование **не доказывает причинность** и не создаёт метку цветения, но помогает выбрать признаки и правила для первой модели.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
FIGSIZE_WIDE = (13, 5)
FAI_CLEAR_WATER_THRESHOLD = 0.60

# Отдельно сохраняем все строки и рабочую выборку для анализа FAI.
eda_all = merged.copy()
eda = (
    merged
    .loc[
        merged['usable_for_fai'].fillna(False)
        & (merged['clear_water_frac'] >= FAI_CLEAR_WATER_THRESHOLD)
        & merged['FAI_mean'].notna()
    ]
    .copy()
)

print(f'Всего спутниковых строк: {len(eda_all):,}')
print(f'Пригодно для анализа FAI: {len(eda):,} ({len(eda) / len(eda_all):.1%})')
print(f'Порог чистой водной поверхности: {FAI_CLEAR_WATER_THRESHOLD:.0%}')

## 5. Покрытие и качество спутниковых наблюдений

Сначала проверяем, сколько доступно снимков каждого сенсора и какая доля из них пригодна для расчёта FAI. Низкая доля обычно означает облака, дымку или недостаточно открытую водную поверхность.

In [ ]:
coverage = (
    eda_all
    .assign(usable_for_fai=lambda x: x['usable_for_fai'].fillna(False))
    .groupby(['year', 'sensor'], as_index=False)
    .agg(
        observations=('date', 'size'),
        usable_observations=('usable_for_fai', 'sum'),
        median_clear_water_frac=('clear_water_frac', 'median'),
    )
)
coverage['usable_share'] = coverage['usable_observations'] / coverage['observations']
display(coverage.sort_values(['year', 'sensor']))

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
for sensor, group in coverage.groupby('sensor'):
    ax.plot(group['year'], group['usable_observations'], marker='o', linewidth=2, label=sensor)
ax.set(title='Пригодные наблюдения FAI по годам', xlabel='Год', ylabel='Число снимков')
ax.legend(title='Сенсор')
plt.show()

## 6. Распределение FAI

FAI может иметь редкие высокие значения. Поэтому рядом с обычным распределением показываем значения в пределах 1–99 процентилей: это делает видимой основную массу наблюдений, не выбрасывая выбросы из таблицы.

In [ ]:
fai_q01, fai_q99 = eda['FAI_mean'].quantile([0.01, 0.99])
fai_summary = eda['FAI_mean'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame('FAI_mean')
display(fai_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(eda['FAI_mean'], bins=35, color='#2a7f62', edgecolor='white')
axes[0].set(title='FAI: все пригодные наблюдения', xlabel='FAI_mean', ylabel='Число снимков')
axes[1].hist(eda.loc[eda['FAI_mean'].between(fai_q01, fai_q99), 'FAI_mean'], bins=35, color='#2a7f62', edgecolor='white')
axes[1].set(title='FAI: диапазон 1–99 процентилей', xlabel='FAI_mean', ylabel='Число снимков')
plt.tight_layout()
plt.show()

## 7. Сезонность и динамика

Графики показывают, меняется ли спутниковый сигнал в течение сезона и в какие даты возникают высокие значения. Точки — отдельные пригодные снимки; линия — медиана FAI по месяцам.

In [ ]:
month_names = {5: 'Май', 6: 'Июн', 7: 'Июл', 8: 'Авг', 9: 'Сен'}
monthly_fai = (
    eda.groupby('month', as_index=False)
    .agg(observations=('FAI_mean', 'size'), median_fai=('FAI_mean', 'median'), mean_fai=('FAI_mean', 'mean'))
)
monthly_fai['month_name'] = monthly_fai['month'].map(month_names)
display(monthly_fai)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
month_values = [eda.loc[eda['month'] == month, 'FAI_mean'].dropna() for month in sorted(month_names)]
axes[0].boxplot(month_values, labels=[month_names[m] for m in sorted(month_names)], showfliers=False)
axes[0].set(title='FAI по месяцам', xlabel='Месяц', ylabel='FAI_mean')

axes[1].scatter(eda['date'], eda['FAI_mean'], s=18, alpha=0.45, color='#2a7f62', label='Отдельный снимок')
monthly_time = eda.set_index('date')['FAI_mean'].resample('MS').median()
axes[1].plot(monthly_time.index, monthly_time.values, color='#cf5c36', linewidth=2.2, label='Медиана за месяц')
axes[1].set(title='Динамика FAI', xlabel='Дата', ylabel='FAI_mean')
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Связь FAI и метеоусловий

Для стартовой модели особенно интересны накопленные условия **до даты снимка**, например средняя температура воды за предыдущие 7 дней. Ниже — ранговые корреляции Спирмена: они ловят монотонную связь и меньше зависят от отдельных выбросов, чем обычная корреляция Пирсона.

In [ ]:
weather_for_eda = [
    'air_temp_c', 'lake_mix_temp_c', 'lake_total_temp_c', 'wind_speed_10m_ms',
    'precipitation_mm', 'solar_radiation_mj_m2',
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d',
    'precipitation_sum_prev_7d', 'wind_mean_prev_7d',
    'calm_days_prev_7d', 'solar_sum_prev_7d',
    'air_temp_mean_prev_14d', 'lake_temp_mean_prev_14d',
    'air_temp_mean_prev_30d', 'lake_temp_mean_prev_30d',
]
weather_for_eda = [column for column in weather_for_eda if column in eda.columns]
spearman = (
    eda[['FAI_mean', *weather_for_eda]]
    .corr(method='spearman')['FAI_mean']
    .drop('FAI_mean')
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename('spearman_r')
    .to_frame()
)
display(spearman)

fig, ax = plt.subplots(figsize=(10, 6))
plot_values = spearman.sort_values('spearman_r')
colors = np.where(plot_values['spearman_r'] >= 0, '#2a7f62', '#cf5c36')
ax.barh(plot_values.index, plot_values['spearman_r'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Связь метеопоказателей с FAI', xlabel='Корреляция Спирмена с FAI_mean', ylabel='Показатель')
plt.tight_layout()
plt.show()

In [ ]:
scatter_features = [
    'lake_temp_mean_prev_7d', 'air_temp_mean_prev_7d',
    'calm_days_prev_7d', 'precipitation_sum_prev_7d',
]
scatter_features = [column for column in scatter_features if column in eda.columns]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, feature in zip(axes.flat, scatter_features):
    data = eda[[feature, 'FAI_mean']].dropna()
    ax.scatter(data[feature], data['FAI_mean'], alpha=0.45, s=20, color='#2a7f62')
    if len(data) >= 3:
        slope, intercept = np.polyfit(data[feature], data['FAI_mean'], 1)
        x_line = np.linspace(data[feature].min(), data[feature].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='#cf5c36', linewidth=2)
    ax.set(title=f'FAI и {feature}', xlabel=feature, ylabel='FAI_mean')
for ax in axes.flat[len(scatter_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 9. Вывод для следующего шага

После просмотра графиков можно переходить к определению целевого сигнала: например, считать потенциально «цветущими» только качественные снимки с FAI выше порога, выбранного по распределению и ручной проверке нескольких сцен. Затем модель должна получать только признаки, доступные **до** прогнозируемой даты.

Не стоит объявлять цветение только по одному FAI: это пока индикатор, который нужно откалибровать по снимкам и, если найдутся, по полевым данным.

## 10. Предварительная метка потенциального цветения

Для каждого сенсора порог определяется как 90-й перцентиль `FAI_mean` среди качественных снимков. Так мы выделяем верхние 10% сигналов **внутри каждого спутника**, а не сравниваем напрямую разные шкалы Landsat и Sentinel-2.

Метка `potential_bloom_signal` имеет три состояния:

- `True` — сильный спутниковый сигнал, кандидат на ручную проверку;
- `False` — качественный снимок, не вошедший в верхние 10%;
- пустое значение — снимок недостаточного качества, его нельзя считать отрицательным примером.

In [ ]:
LABEL_QUANTILE = 0.90
LABELED_OUTPUT_FILE = Path('shershnevskoe_satellite_weather_labeled.csv')
CANDIDATES_OUTPUT_FILE = Path('potential_bloom_candidates_for_review.csv')

labeled = merged.copy()
is_quality_fai = (
    labeled['usable_for_fai'].fillna(False)
    & (labeled['clear_water_frac'] >= FAI_CLEAR_WATER_THRESHOLD)
    & labeled['FAI_mean'].notna()
)

# Порог пересчитывается после каждой новой выгрузки и отдельно для каждого сенсора.
threshold_by_sensor = (
    labeled.loc[is_quality_fai]
    .groupby('sensor')['FAI_mean']
    .quantile(LABEL_QUANTILE)
    .rename('fai_threshold_p90_by_sensor')
)
labeled = labeled.join(threshold_by_sensor, on='sensor')

labeled['fai_percentile_within_sensor'] = np.nan
labeled.loc[is_quality_fai, 'fai_percentile_within_sensor'] = (
    labeled.loc[is_quality_fai]
    .groupby('sensor')['FAI_mean']
    .rank(pct=True, method='average')
)

labeled['potential_bloom_signal'] = pd.Series(pd.NA, index=labeled.index, dtype='boolean')
labeled.loc[is_quality_fai, 'potential_bloom_signal'] = (
    labeled.loc[is_quality_fai, 'FAI_mean']
    >= labeled.loc[is_quality_fai, 'fai_threshold_p90_by_sensor']
)

display(threshold_by_sensor.reset_index())

label_summary = (
    labeled.loc[is_quality_fai]
    .groupby('sensor', as_index=False)
    .agg(
        quality_observations=('date', 'size'),
        potential_bloom_candidates=('potential_bloom_signal', 'sum'),
        candidate_share=('potential_bloom_signal', 'mean'),
    )
)
display(label_summary)

In [ ]:
review_columns = [
    'date', 'sensor', 'FAI_mean', 'FAI_p90', 'FAI_p95',
    'fai_threshold_p90_by_sensor', 'fai_percentile_within_sensor',
    'clear_water_frac', 'air_temp_c', 'lake_mix_temp_c',
    'lake_temp_mean_prev_7d', 'precipitation_sum_prev_7d', 'calm_days_prev_7d',
]
review_columns = [column for column in review_columns if column in labeled.columns]
candidates = (
    labeled.loc[labeled['potential_bloom_signal'].fillna(False), review_columns]
    .sort_values(['sensor', 'FAI_mean'], ascending=[True, False])
    .copy()
)
display(candidates)

labeled_to_save = labeled.sort_values(['date', 'sensor']).copy()
labeled_to_save['date'] = pd.to_datetime(labeled_to_save['date']).dt.strftime('%Y-%m-%d')
labeled_to_save.to_csv(LABELED_OUTPUT_FILE, index=False)
candidates.to_csv(CANDIDATES_OUTPUT_FILE, index=False)

print(f'Сохранена размеченная база: {LABELED_OUTPUT_FILE.resolve()}')
print(f'Сохранён список кандидатов для проверки: {CANDIDATES_OUTPUT_FILE.resolve()}')
print(f'Кандидатов на потенциальное цветение: {len(candidates)}')

## 11. Шаблон ручной проверки

Для проверки подготовлены три группы снимков: 10 с наиболее высоким FAI, 10 ближе всего к порогу и 5 с низким FAI для сравнения. Для каждой строки в CSV нужно заполнить `manual_label` и, при необходимости, `manual_comment`.

Допустимые значения `manual_label`: `bloom`, `no_bloom`, `shore_vegetation`, `turbidity`, `clouds_or_artifact`.

Файл шаблона создаётся только один раз и при повторном запуске ноутбука не перезаписывается, чтобы сохранить внесённые вручную решения.

In [ ]:
MANUAL_REVIEW_FILE = Path('manual_bloom_validation_template.csv')

quality_for_review = labeled.loc[is_quality_fai].copy()
quality_for_review['distance_to_threshold'] = (
    quality_for_review['FAI_mean'] - quality_for_review['fai_threshold_p90_by_sensor']
).abs()

top_fai = quality_for_review.nlargest(10, 'FAI_mean').assign(review_priority='highest_fai')
near_threshold = quality_for_review.nsmallest(10, 'distance_to_threshold').assign(review_priority='near_threshold')
low_fai = quality_for_review.nsmallest(5, 'FAI_mean').assign(review_priority='low_fai_control')

review_set = pd.concat([top_fai, near_threshold, low_fai], ignore_index=True)
review_set = review_set.drop_duplicates(subset=['date', 'sensor'], keep='first')
review_set = review_set.sort_values(['review_priority', 'sensor', 'FAI_mean'], ascending=[True, True, False]).copy()

manual_columns = [
    'date', 'sensor', 'review_priority', 'FAI_mean', 'FAI_p90', 'FAI_p95',
    'fai_threshold_p90_by_sensor', 'fai_percentile_within_sensor',
    'clear_water_frac', 'air_temp_c', 'lake_mix_temp_c',
    'lake_temp_mean_prev_7d', 'precipitation_sum_prev_7d', 'calm_days_prev_7d',
]
manual_columns = [column for column in manual_columns if column in review_set.columns]
manual_template = review_set[manual_columns].copy()
manual_template['manual_label'] = pd.NA
manual_template['manual_comment'] = pd.NA
manual_template['manual_reviewed'] = False
manual_template['date'] = pd.to_datetime(manual_template['date']).dt.strftime('%Y-%m-%d')

if MANUAL_REVIEW_FILE.exists():
    print(f'Шаблон уже существует и не был перезаписан: {MANUAL_REVIEW_FILE.resolve()}')
else:
    manual_template.to_csv(MANUAL_REVIEW_FILE, index=False)
    print(f'Создан шаблон для ручной проверки: {MANUAL_REVIEW_FILE.resolve()}')

display(manual_template)

## 12. Как проверить строку в Google Earth Engine

Открой файл `gee_manual_review.js`, скопируй его в [Google Earth Engine Code Editor](https://code.earthengine.google.com/) и подставь вверху `date` и `sensor` из строки `manual_bloom_validation_template.csv`.

Сначала включена естественная цветопередача. Затем при необходимости включи слой `false colour (NIR-R-G)`: зелёные/яркие участки на воде там виднее, но прибрежную растительность нужно не путать с цветением.

# Подготовка данных и базовая модель на 7 дней

Эта часть использует ручные метки как приоритетные. Для Landsat качественные, но не размеченные вручную снимки получают **слабую метку** по проверенному правилу FAI. Sentinel-2 без ручной метки здесь исключён: проверка показала, что один FAI у него недостаточно надёжен.

Горизонт модели — 7 дней: для снимка на дату T она берёт метеопризнаки на T−7, то есть будущая информация не попадает в признаки.

In [ ]:
MANUAL_REVIEW_FILE = Path('manual_bloom_validation_template.csv')
if not MANUAL_REVIEW_FILE.exists():
    raise FileNotFoundError(f'Не найден файл ручной разметки: {MANUAL_REVIEW_FILE.resolve()}')

manual = pd.read_csv(MANUAL_REVIEW_FILE)
manual['date'] = pd.to_datetime(manual['date']).dt.normalize()
manual['manual_label'] = manual['manual_label'].fillna('').str.strip()
manual['manual_reviewed'] = manual['manual_reviewed'].fillna(False).astype(bool)

allowed_manual_labels = {'bloom', 'no_bloom', 'clouds_or_artifact', 'shore_vegetation', 'turbidity'}
unknown_labels = set(manual.loc[manual['manual_reviewed'], 'manual_label']) - allowed_manual_labels
if unknown_labels:
    raise ValueError(f'Неизвестные ручные метки: {unknown_labels}')

manual_checked = manual.loc[manual['manual_reviewed']].copy()
manual_for_merge = manual_checked[['date', 'sensor', 'manual_label', 'manual_comment']].copy()
if manual_for_merge.duplicated(['date', 'sensor']).any():
    raise ValueError('В ручной разметке повторяется пара date + sensor.')

calibrated = labeled.merge(manual_for_merge, on=['date', 'sensor'], how='left', validate='one_to_one')
manual_summary = (
    manual_checked.groupby(['sensor', 'manual_label'], as_index=False)
    .size().rename(columns={'size': 'n'})
)
display(manual_summary)

## 13. Калибровка FAI после ручной проверки

Для Landsat берём предварительный порог `0.007`: все вручную подтверждённые случаи цветения около этого диапазона и выше, а уверенные отрицательные случаи существенно ниже. Это рабочее правило для хакатона, не биологический универсальный норматив.

Для Sentinel-2 не распространяем FAI-порог на неразмеченные сцены: среди вручную просмотренных высоких значений были и `bloom`, и `no_bloom`.

In [ ]:
LANDSAT_WEAK_LABEL_THRESHOLD = 0.007

manual_fai_summary = (
    calibrated.loc[calibrated['manual_label'].isin(['bloom', 'no_bloom'])]
    .groupby(['sensor', 'manual_label'], as_index=False)
    .agg(n=('FAI_mean', 'size'), min_fai=('FAI_mean', 'min'), median_fai=('FAI_mean', 'median'), max_fai=('FAI_mean', 'max'))
)
display(manual_fai_summary)

fig, ax = plt.subplots(figsize=(10, 4.5))
for (sensor, label), group in calibrated.loc[calibrated['manual_label'].isin(['bloom', 'no_bloom'])].groupby(['sensor', 'manual_label']):
    color = '#2a7f62' if label == 'bloom' else '#cf5c36'
    ax.scatter([sensor] * len(group), group['FAI_mean'], s=65, alpha=0.8, color=color, label=f'{sensor}: {label}')
ax.axhline(LANDSAT_WEAK_LABEL_THRESHOLD, color='black', linestyle='--', label='Порог Landsat = 0.007')
ax.set(title='Ручные метки и FAI', xlabel='Сенсор', ylabel='FAI_mean')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 14. Датасет для модели

Базовая модель обучается только на Landsat. Цель — `bloom_target_7d`: наличие цветения в дату снимка T. Признаки берутся из погоды на дату T−7. Ручные метки получают больший вес при обучении, а слабые метки по FAI используются только как дополнение.

In [ ]:
MODEL_DATASET_FILE = Path('shershnevskoe_model_dataset_7d_landsat.csv')

landsat_model = calibrated.loc[
    (calibrated['sensor'] == 'LANDSAT_8_9')
    & calibrated['usable_for_fai'].fillna(False)
    & (calibrated['clear_water_frac'] >= FAI_CLEAR_WATER_THRESHOLD)
    & calibrated['FAI_mean'].notna()
].copy()

landsat_model['bloom_target_7d'] = pd.Series(pd.NA, index=landsat_model.index, dtype='Int64')
landsat_model['label_source'] = pd.NA
manual_positive = landsat_model['manual_label'].eq('bloom')
manual_negative = landsat_model['manual_label'].eq('no_bloom')
manual_unusable = landsat_model['manual_label'].isin(['clouds_or_artifact', 'shore_vegetation', 'turbidity'])

landsat_model.loc[manual_positive, 'bloom_target_7d'] = 1
landsat_model.loc[manual_positive, 'label_source'] = 'manual'
landsat_model.loc[manual_negative, 'bloom_target_7d'] = 0
landsat_model.loc[manual_negative, 'label_source'] = 'manual'
weak_label_rows = landsat_model['manual_label'].isna() & ~manual_unusable
landsat_model.loc[weak_label_rows, 'bloom_target_7d'] = (
    landsat_model.loc[weak_label_rows, 'FAI_mean'] >= LANDSAT_WEAK_LABEL_THRESHOLD
).astype('Int64')
landsat_model.loc[weak_label_rows, 'label_source'] = 'weak_fai_rule'

landsat_model['issue_date'] = landsat_model['date'] - pd.Timedelta(days=7)
issue_weather_features = [
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d',
    'precipitation_sum_prev_7d', 'wind_mean_prev_7d',
    'calm_days_prev_7d', 'solar_sum_prev_7d',
    'air_temp_mean_prev_14d', 'lake_temp_mean_prev_14d',
    'air_temp_mean_prev_30d', 'lake_temp_mean_prev_30d',
]
issue_weather_features = [column for column in issue_weather_features if column in weather.columns]
# Префикс issue_ защищает от случайного использования погоды в дату снимка T.
weather_at_issue = weather[['date', *issue_weather_features]].copy()
weather_at_issue = weather_at_issue.rename(columns={
    'date': 'issue_date',
    **{column: f'issue_{column}' for column in issue_weather_features},
})
model_data = landsat_model.merge(weather_at_issue, on='issue_date', how='left', validate='many_to_one')
model_data['target_year'] = model_data['date'].dt.year
model_data['target_month'] = model_data['date'].dt.month
model_data['target_month_sin'] = np.sin(2 * np.pi * model_data['target_month'] / 12)
model_data['target_month_cos'] = np.cos(2 * np.pi * model_data['target_month'] / 12)

feature_columns = [f'issue_{column}' for column in issue_weather_features] + [
    'target_month_sin', 'target_month_cos',
]
model_data = model_data.dropna(subset=['bloom_target_7d', *feature_columns]).copy()
model_data['bloom_target_7d'] = model_data['bloom_target_7d'].astype(int)

dataset_summary = model_data.groupby(['label_source', 'bloom_target_7d'], as_index=False).size().rename(columns={'size': 'n'})
display(dataset_summary)

model_data_to_save = model_data.sort_values('date').copy()
for date_column in ['date', 'issue_date']:
    model_data_to_save[date_column] = model_data_to_save[date_column].dt.strftime('%Y-%m-%d')
model_data_to_save.to_csv(MODEL_DATASET_FILE, index=False)
print(f'Сохранён датасет модели: {MODEL_DATASET_FILE.resolve()}')

## 15. Базовая вероятностная модель

Используем простую логистическую регрессию с балансировкой классов. Проверка идёт по времени: более ранние годы — обучение, более поздние — тест. Метрики на таком малом наборе — ориентир, а не доказательство точности.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

years = sorted(model_data['target_year'].unique())
valid_splits = []
for test_start_year in years[1:]:
    train_part = model_data.loc[model_data['target_year'] < test_start_year]
    test_part = model_data.loc[model_data['target_year'] >= test_start_year]
    if len(test_part) >= 10 and train_part['bloom_target_7d'].nunique() == 2 and test_part['bloom_target_7d'].nunique() == 2:
        valid_splits.append(test_start_year)

if not valid_splits:
    raise ValueError('Недостаточно положительных и отрицательных примеров для временной проверки модели.')

TEST_START_YEAR = valid_splits[-1]
train_data = model_data.loc[model_data['target_year'] < TEST_START_YEAR].copy()
test_data = model_data.loc[model_data['target_year'] >= TEST_START_YEAR].copy()

baseline_model = Pipeline([
    ('scale', StandardScaler()),
    ('logistic', LogisticRegression(class_weight='balanced', max_iter=2_000, random_state=42)),
])
train_weights = np.where(train_data['label_source'].eq('manual'), 3.0, 1.0)
baseline_model.fit(
    train_data[feature_columns],
    train_data['bloom_target_7d'],
    logistic__sample_weight=train_weights,
)

test_probability = baseline_model.predict_proba(test_data[feature_columns])[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
metrics = pd.DataFrame([{
    'test_start_year': TEST_START_YEAR,
    'train_rows': len(train_data),
    'test_rows': len(test_data),
    'test_bloom_share': test_data['bloom_target_7d'].mean(),
    'roc_auc': roc_auc_score(test_data['bloom_target_7d'], test_probability),
    'average_precision': average_precision_score(test_data['bloom_target_7d'], test_probability),
    'balanced_accuracy_at_0_5': balanced_accuracy_score(test_data['bloom_target_7d'], test_prediction),
}])
display(metrics)
print('Матрица ошибок:')
display(pd.DataFrame(
    confusion_matrix(test_data['bloom_target_7d'], test_prediction),
    index=['fact_no_bloom', 'fact_bloom'],
    columns=['pred_no_bloom', 'pred_bloom'],
))

coefficients = pd.DataFrame({
    'feature': feature_columns,
    'standardized_coefficient': baseline_model.named_steps['logistic'].coef_[0],
}).sort_values('standardized_coefficient', key=lambda values: values.abs(), ascending=False)
display(coefficients)

## 16. Историческая ежедневная оценка риска

Модель дообучается на всём имеющемся Landsat-датасете и считает историческую оценку риска на каждый день, для которого в ERA5-Land есть нужная погодная история. Это **бэктест**, а не прогноз на будущую неделю: для реального будущего прогноза в те же признаки нужно подать прогноз погоды.

In [ ]:
HISTORICAL_RISK_FILE = Path('shershnevskoe_historical_bloom_risk_7d.csv')

final_model = clone(baseline_model)
full_weights = np.where(model_data['label_source'].eq('manual'), 3.0, 1.0)
final_model.fit(
    model_data[feature_columns],
    model_data['bloom_target_7d'],
    logistic__sample_weight=full_weights,
)

daily_risk = weather.copy()
daily_risk['issue_date'] = daily_risk['date']
daily_risk['target_date'] = daily_risk['issue_date'] + pd.Timedelta(days=7)
daily_risk['target_month'] = daily_risk['target_date'].dt.month
daily_risk['target_month_sin'] = np.sin(2 * np.pi * daily_risk['target_month'] / 12)
daily_risk['target_month_cos'] = np.cos(2 * np.pi * daily_risk['target_month'] / 12)
for column in issue_weather_features:
    daily_risk[f'issue_{column}'] = daily_risk[column]
daily_risk = daily_risk.dropna(subset=feature_columns).copy()
daily_risk['bloom_probability_7d'] = final_model.predict_proba(daily_risk[feature_columns])[:, 1]
daily_risk['risk_level'] = pd.cut(
    daily_risk['bloom_probability_7d'],
    bins=[-0.01, 0.30, 0.60, 1.0],
    labels=['low', 'medium', 'high'],
)

risk_to_save = daily_risk[['issue_date', 'target_date', 'bloom_probability_7d', 'risk_level']].copy()
for date_column in ['issue_date', 'target_date']:
    risk_to_save[date_column] = risk_to_save[date_column].dt.strftime('%Y-%m-%d')
risk_to_save.to_csv(HISTORICAL_RISK_FILE, index=False)

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
ax.plot(daily_risk['target_date'], daily_risk['bloom_probability_7d'], color='#2a7f62', linewidth=1)
ax.set(title='Историческая оценка риска цветения на 7 дней', xlabel='Прогнозируемая дата', ylabel='Вероятность')
ax.set_ylim(0, 1)
plt.show()

print(f'Сохранён исторический бэктест риска: {HISTORICAL_RISK_FILE.resolve()}')

# Наглядная проверка train/test и метрик

Разбиение только временное: модель учится на более ранних годах и проверяется на более поздних. Так мы не используем будущее для обучения. В тестовой части показаны все строки, поэтому можно вручную сопоставить фактическую метку и прогноз модели.

In [ ]:
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, precision_score, recall_score

TRAIN_SPLIT_FILE = Path('shershnevskoe_train_split_7d.csv')
TEST_SPLIT_FILE = Path('shershnevskoe_test_split_7d.csv')

# В train попадают годы до TEST_START_YEAR, в test — TEST_START_YEAR и позднее.
train_overview = (
    train_data.groupby(['target_year', 'bloom_target_7d'], as_index=False)
    .size().rename(columns={'size': 'observations'})
    .assign(split='train')
)
test_overview = (
    test_data.groupby(['target_year', 'bloom_target_7d'], as_index=False)
    .size().rename(columns={'size': 'observations'})
    .assign(split='test')
)
split_overview = pd.concat([train_overview, test_overview], ignore_index=True)
display(split_overview.sort_values(['split', 'target_year', 'bloom_target_7d']))

test_eval = test_data[['date', 'issue_date', 'sensor', 'FAI_mean', 'bloom_target_7d', 'label_source']].copy()
test_eval['predicted_probability_7d'] = test_probability
test_eval['predicted_label_at_0_5'] = test_prediction
test_eval['is_correct_at_0_5'] = test_eval['bloom_target_7d'].eq(test_eval['predicted_label_at_0_5'])
display(test_eval.sort_values('date'))

train_to_save = train_data.sort_values('date').copy()
test_to_save = test_eval.sort_values('date').copy()
for frame in [train_to_save, test_to_save]:
    for date_column in ['date', 'issue_date']:
        frame[date_column] = pd.to_datetime(frame[date_column]).dt.strftime('%Y-%m-%d')
train_to_save.to_csv(TRAIN_SPLIT_FILE, index=False)
test_to_save.to_csv(TEST_SPLIT_FILE, index=False)

metric_table = pd.DataFrame([{
    'Метрика': 'ROC-AUC',
    'Значение': roc_auc_score(test_data['bloom_target_7d'], test_probability),
    'Что показывает': 'Насколько модель ставит цветению вероятность выше, чем отсутствию цветения.',
}, {
    'Метрика': 'Average precision',
    'Значение': average_precision_score(test_data['bloom_target_7d'], test_probability),
    'Что показывает': 'Качество ранжирования редких положительных случаев.',
}, {
    'Метрика': 'Balanced accuracy при пороге 0.5',
    'Значение': balanced_accuracy_score(test_data['bloom_target_7d'], test_prediction),
    'Что показывает': 'Средняя точность для классов bloom и no_bloom без перекоса в частый класс.',
}, {
    'Метрика': 'Precision при пороге 0.5',
    'Значение': precision_score(test_data['bloom_target_7d'], test_prediction, zero_division=0),
    'Что показывает': 'Какая доля тревог модели действительно оказалась цветением.',
}, {
    'Метрика': 'Recall при пороге 0.5',
    'Значение': recall_score(test_data['bloom_target_7d'], test_prediction, zero_division=0),
    'Что показывает': 'Какую долю реальных цветений модель нашла.',
}, {
    'Метрика': 'F1 при пороге 0.5',
    'Значение': f1_score(test_data['bloom_target_7d'], test_prediction, zero_division=0),
    'Что показывает': 'Баланс precision и recall.',
}, {
    'Метрика': 'Brier score',
    'Значение': brier_score_loss(test_data['bloom_target_7d'], test_probability),
    'Что показывает': 'Насколько хорошо откалиброваны сами вероятности. Меньше — лучше.',
}, {
    'Метрика': 'Обычная accuracy при пороге 0.5',
    'Значение': accuracy_score(test_data['bloom_target_7d'], test_prediction),
    'Что показывает': 'Общая доля верных ответов. При редком цветении вторична.',
}])
display(metric_table.style.format({'Значение': '{:.3f}'}))
print(f'Временной разрез: train — до {TEST_START_YEAR}, test — с {TEST_START_YEAR}.')
print(f'Сохранены: {TRAIN_SPLIT_FILE.resolve()} и {TEST_SPLIT_FILE.resolve()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (split_name, data) in zip(axes, [('train', train_data), ('test', test_data)]):
    counts = data.groupby(['target_year', 'bloom_target_7d']).size().unstack(fill_value=0)
    counts = counts.reindex(columns=[0, 1], fill_value=0)
    ax.bar(counts.index, counts[0], color='#8aa6b5', label='no_bloom (0)')
    ax.bar(counts.index, counts[1], bottom=counts[0], color='#2a7f62', label='bloom (1)')
    ax.set(title=f'{split_name}: наблюдения по годам', xlabel='Год', ylabel='Число строк')
    ax.legend()
axes[1].scatter(
    test_eval['date'], test_eval['predicted_probability_7d'],
    c=np.where(test_eval['bloom_target_7d'].eq(1), '#2a7f62', '#cf5c36'),
    s=70, alpha=0.85, label='Точка: факт (зелёный=bloom, оранжевый=no_bloom)'
)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1, label='Порог решения 0.5')
axes[1].set(title='Тест: прогнозы и фактические метки', xlabel='Дата снимка', ylabel='Вероятность bloom через 7 дней', ylim=(0, 1))
axes[1].legend(loc='best')
plt.tight_layout()
plt.show()

# Средний сезонный год

Ниже одна линия: для каждого месяца усредняется историческая вероятность модели по всем годам. По оси X показаны все 12 месяцев. Для января–апреля и ноября–декабря нет исходных погодных и спутниковых данных в текущем проекте, поэтому там стоит пропуск, а не искусственный ноль. Октябрь показан только потому, что сентябрьская погода даёт прогноз на семь дней вперёд.

In [ ]:
SEASONAL_RISK_FILE = Path('shershnevskoe_average_seasonal_risk_7d.csv')
month_names_ru = {
    1: 'Янв', 2: 'Фев', 3: 'Мар', 4: 'Апр', 5: 'Май', 6: 'Июн',
    7: 'Июл', 8: 'Авг', 9: 'Сен', 10: 'Окт', 11: 'Ноя', 12: 'Дек',
}
monthly_risk = (
    daily_risk.groupby(daily_risk['target_date'].dt.month)['bloom_probability_7d']
    .agg(mean_probability='mean', median_probability='median', days='size')
    .reindex(range(1, 13))
    .rename_axis('month').reset_index()
)
monthly_risk['month_name'] = monthly_risk['month'].map(month_names_ru)
monthly_risk.to_csv(SEASONAL_RISK_FILE, index=False)
display(monthly_risk[['month', 'month_name', 'mean_probability', 'median_probability', 'days']])

fig, ax = plt.subplots(figsize=(13, 5))
available = monthly_risk.dropna(subset=['mean_probability'])
ax.axvspan(0.5, 4.5, color='#e8e8e8', alpha=0.8, label='Нет данных вне сезона')
ax.axvspan(10.5, 12.5, color='#e8e8e8', alpha=0.8)
ax.plot(available['month'], available['mean_probability'], marker='o', linewidth=2.5, color='#2a7f62', label='Средняя вероятность')
ax.scatter(available['month'], available['median_probability'], s=55, color='#cf5c36', label='Медиана вероятности')
ax.set(
    title='Средняя историческая вероятность цветения через 7 дней по месяцам',
    xlabel='Прогнозируемый месяц', ylabel='Вероятность', ylim=(0, 1), xlim=(0.5, 12.5),
)
ax.set_xticks(range(1, 13), [month_names_ru[month] for month in range(1, 13)])
ax.legend(loc='best')
plt.tight_layout()
plt.show()

peak_month = available.loc[available['mean_probability'].idxmax()]
print(f'Наибольшая средняя вероятность в доступных данных: {peak_month.month_name} ({peak_month.mean_probability:.1%}).')
print(f'Сохранён средний сезонный профиль: {SEASONAL_RISK_FILE.resolve()}')

# Экспериментальная общая спутниковая модель

Эта модель сохраняет в признаках FAI, MNDWI и доступные Landsat-индексы, а также использует оба спутника. Её задача отличается от погодной модели: она оценивает вероятность цветения **в день спутникового наблюдения**, когда FAI уже доступен.

Обучение идёт только на вручную размеченных строках `bloom` и `no_bloom`. Все остальные качественные спутниковые наблюдения не становятся псевдометками, а получают только итоговую вероятность после обучения. Это исключает обучение модели на её же FAI-правиле.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder

GENERAL_SCORES_FILE = Path('shershnevskoe_general_satellite_model_scores.csv')

# Общие признаки и Landsat-специфические индексы. Пропуски последних будут отмечены отдельным индикатором.
general_numeric_features = [
    'FAI_mean', 'FAI_p90', 'fai_percentile_within_sensor',
    'MNDWI_mean', 'MNDWI_p90', 'clear_water_frac',
    'Turbidity_FNU_mean', 'SPM_mgL_mean', 'B10_C_mean',
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d',
    'precipitation_sum_prev_7d', 'wind_mean_prev_7d',
    'calm_days_prev_7d', 'solar_sum_prev_7d',
]
general_numeric_features = [column for column in general_numeric_features if column in calibrated.columns]
general_categorical_features = ['sensor']

is_quality_general = (
    calibrated['usable_for_fai'].fillna(False)
    & (calibrated['clear_water_frac'] >= FAI_CLEAR_WATER_THRESHOLD)
    & calibrated['FAI_mean'].notna()
)
general_all = calibrated.loc[is_quality_general].copy()
general_train = general_all.loc[general_all['manual_label'].isin(['bloom', 'no_bloom'])].copy()
general_train['manual_bloom_target'] = general_train['manual_label'].eq('bloom').astype(int)

label_balance = (
    general_train.groupby(['sensor', 'manual_label'], as_index=False)
    .size().rename(columns={'size': 'n'})
)
display(label_balance)
print(f'Качественных наблюдений для оценки: {len(general_all)}')
print(f'Ручных строк для обучения: {len(general_train)}')

## 17. Проверка общей модели

Ручных меток всего 20, поэтому временный test здесь был бы слишком мал. Для честной предварительной оценки используется стратифицированная 5-fold cross-validation: каждая вручную размеченная строка получает прогноз от модели, которая не видела эту строку при обучении. Результат экспериментальный и нужен для сравнения подходов, а не как окончательная оценка точности.

In [ ]:
general_preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler()),
    ]), general_numeric_features),
    ('sensor', OneHotEncoder(handle_unknown='ignore'), general_categorical_features),
])

# L1-регуляризация удерживает в модели только действительно полезные признаки.
general_model = Pipeline([
    ('preprocess', general_preprocess),
    ('logistic', LogisticRegression(
        penalty='l1', solver='liblinear', C=0.5, class_weight='balanced', max_iter=2_000, random_state=42
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
general_oof_probability = cross_val_predict(
    general_model,
    general_train[[*general_numeric_features, *general_categorical_features]],
    general_train['manual_bloom_target'],
    cv=cv, method='predict_proba',
)[:, 1]
general_oof_prediction = (general_oof_probability >= 0.5).astype(int)

general_metrics = pd.DataFrame([{
    'Метрика': 'ROC-AUC (5-fold OOF)',
    'Значение': roc_auc_score(general_train['manual_bloom_target'], general_oof_probability),
}, {
    'Метрика': 'Average precision (5-fold OOF)',
    'Значение': average_precision_score(general_train['manual_bloom_target'], general_oof_probability),
}, {
    'Метрика': 'Balanced accuracy при пороге 0.5',
    'Значение': balanced_accuracy_score(general_train['manual_bloom_target'], general_oof_prediction),
}, {
    'Метрика': 'Precision при пороге 0.5',
    'Значение': precision_score(general_train['manual_bloom_target'], general_oof_prediction, zero_division=0),
}, {
    'Метрика': 'Recall при пороге 0.5',
    'Значение': recall_score(general_train['manual_bloom_target'], general_oof_prediction, zero_division=0),
}])
display(general_metrics.style.format({'Значение': '{:.3f}'}))

general_oof = general_train[['date', 'sensor', 'FAI_mean', 'fai_percentile_within_sensor', 'manual_label']].copy()
general_oof['manual_bloom_target'] = general_train['manual_bloom_target']
general_oof['oof_probability'] = general_oof_probability
general_oof['oof_prediction_at_0_5'] = general_oof_prediction
display(general_oof.sort_values(['sensor', 'date']))
print(classification_report(general_train['manual_bloom_target'], general_oof_prediction, zero_division=0))

In [ ]:
# Финальная подгонка на всех ручных метках и оценка каждого качественного снимка.
general_model.fit(
    general_train[[*general_numeric_features, *general_categorical_features]],
    general_train['manual_bloom_target'],
)
general_all['general_satellite_bloom_probability'] = general_model.predict_proba(
    general_all[[*general_numeric_features, *general_categorical_features]]
)[:, 1]
general_all['general_satellite_risk_level'] = pd.cut(
    general_all['general_satellite_bloom_probability'],
    bins=[-0.01, 0.30, 0.60, 1.0], labels=['low', 'medium', 'high'],
)

transformed_feature_names = general_model.named_steps['preprocess'].get_feature_names_out()
general_coefficients = pd.DataFrame({
    'feature': transformed_feature_names,
    'coefficient': general_model.named_steps['logistic'].coef_[0],
})
general_coefficients = general_coefficients.loc[general_coefficients['coefficient'].abs() > 1e-8]
general_coefficients = general_coefficients.sort_values('coefficient', key=lambda values: values.abs(), ascending=False)
display(general_coefficients)

general_scores_to_save = general_all.sort_values(['date', 'sensor']).copy()
general_scores_to_save['date'] = general_scores_to_save['date'].dt.strftime('%Y-%m-%d')
general_scores_to_save.to_csv(GENERAL_SCORES_FILE, index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for sensor, group in general_all.groupby('sensor'):
    axes[0].scatter(
        group['FAI_mean'], group['general_satellite_bloom_probability'],
        s=22, alpha=0.55, label=sensor,
    )
axes[0].set(title='FAI и оценка общей модели', xlabel='FAI_mean', ylabel='Вероятность цветения', ylim=(0, 1))
axes[0].legend()

manual_plot = general_all.loc[general_all['manual_label'].isin(['bloom', 'no_bloom'])]
colors = np.where(manual_plot['manual_label'].eq('bloom'), '#2a7f62', '#cf5c36')
axes[1].scatter(manual_plot['date'], manual_plot['general_satellite_bloom_probability'], c=colors, s=75, alpha=0.9)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1)
axes[1].set(title='Ручные метки и оценка общей модели', xlabel='Дата снимка', ylabel='Вероятность цветения', ylim=(0, 1))
plt.tight_layout()
plt.show()

print(f'Сохранены оценки общей модели: {GENERAL_SCORES_FILE.resolve()}')